In [1]:
import pandas as pd

# Load
ratings = pd.read_csv('Ratings.csv')
books   = pd.read_csv('Books.csv', low_memory=False)

# Fix column names
ratings.columns = ['user_id', 'isbn', 'rating']
books.columns   = ['isbn', 'title', 'author', 'year', 
                   'publisher', 'img_s', 'img_m', 'img_l']

print("Ratings:", ratings.shape)
print("Books:  ", books.shape)
print("✓ loaded")


Ratings: (1149780, 3)
Books:   (271360, 8)
✓ loaded


In [2]:
# Drop implicit ratings (0 = user saw book but didn't rate)
ratings = ratings[ratings['rating'] > 0]

print("After dropping zeros:", ratings.shape)
print("Rating value counts:")
print(ratings['rating'].value_counts().sort_index())

After dropping zeros: (433671, 3)
Rating value counts:
rating
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64


In [3]:
# Count ratings per book + calculate average
popularity = ratings.groupby('isbn').agg(
    num_ratings = ('rating', 'count'),
    avg_rating  = ('rating', 'mean')
).reset_index()

print("Total books with ratings:", len(popularity))
print("\nSample:")
print(popularity.head())
print("\nDistribution of rating counts:")
print(popularity['num_ratings'].describe().round(2))

Total books with ratings: 185973

Sample:
           isbn  num_ratings  avg_rating
0    0330299891            1         6.0
1    0375404120            1         3.0
2    9022906116            1         7.0
3      #6612432            1         5.0
4  '9607092910'            1        10.0

Distribution of rating counts:
count    185973.00
mean          2.33
std           6.83
min           1.00
25%           1.00
50%           1.00
75%           2.00
max         707.00
Name: num_ratings, dtype: float64


In [4]:
# Filter — only keep books with enough ratings to be meaningful
# 50+ ratings AND average rating above 7
popularity = popularity[
    (popularity['num_ratings'] >= 50) &
    (popularity['avg_rating']  >= 7)
]

print("Books after filter:", len(popularity))
print("\nTop 10 by number of ratings:")
print(popularity.sort_values('num_ratings', ascending=False).head(10))

Books after filter: 512

Top 10 by number of ratings:
             isbn  num_ratings  avg_rating
26378  0316666343          707    8.185290
44961  0385504209          487    8.435318
22405  0312195516          383    8.182768
90207  0679781587          333    8.408408
5269   0060928336          320    7.887500
77942  059035342X          313    8.939297
15990  0142001740          307    8.452769
58853  0446672211          295    8.142373
54839  044023722X          281    7.338078
64940  0452282152          278    7.982014


In [5]:
# Merge with books to get actual titles and authors
popularity = popularity.merge(
    books[['isbn', 'title', 'author', 'img_m']], 
    on='isbn', 
    how='left'
)

# Drop books where title didn't match (ISBN not in Books.csv)
popularity = popularity.dropna(subset=['title'])

# Sort by number of ratings
popularity = popularity.sort_values('num_ratings', ascending=False)

print("Final trending books:", len(popularity))
print("\nTop 10 trending:")
print(popularity[['title', 'author', 'num_ratings', 'avg_rating']].head(10))

Final trending books: 506

Top 10 trending:
                                                 title           author  \
87                           The Lovely Bones: A Novel     Alice Sebold   
196                                  The Da Vinci Code        Dan Brown   
58                 The Red Tent (Bestselling Backlist)    Anita Diamant   
12     Divine Secrets of the Ya-Ya Sisterhood: A Novel    Rebecca Wells   
395  Harry Potter and the Sorcerer's Stone (Harry P...    J. K. Rowling   
51                             The Secret Life of Bees    Sue Monk Kidd   
307  Where the Heart Is (Oprah's Book Club (Paperba...     Billie Letts   
263                                    A Painted House     John Grisham   
339                          Girl with a Pearl Earring  Tracy Chevalier   
84                          The Pilot's Wife : A Novel     Anita Shreve   

     num_ratings  avg_rating  
87           707    8.185290  
196          487    8.435318  
58           383    8.182768  
12    

In [6]:
import pickle

# Round avg_rating to 2 decimal places
popularity['avg_rating'] = popularity['avg_rating'].round(2)

# Save as CSV for P3 to load in the app
popularity.to_csv('top_books.csv', index=False)

# Also save as pickle for Flask API
pickle.dump(popularity, open('popularity_model.pkl', 'wb'))

print(f"Saved {len(popularity)} trending books")
print("✓ top_books.csv")
print("✓ popularity_model.pkl")

Saved 506 trending books
✓ top_books.csv
✓ popularity_model.pkl
